# 03 — Modeling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/content/drive/MyDrive/BSE405_SoilMoisture/data'
RESULTS_DIR = '/content/drive/MyDrive/BSE405_SoilMoisture/results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

df = pd.read_csv(f'{DATA_DIR}/processed_data.csv')
print(f"Shape: {df.shape}")
df.head()

## Define Features and Models

In [ ]:
FEATURE_COLS = ['latitude', 'longitude', 'clay_content', 'sand_content', 'silt_content',
                'sm_aux', 'day_of_year', 'clay_sand_ratio', 'clay_silt_ratio', 'clay_x_sm_aux']

df = pd.get_dummies(df, columns=['season'], drop_first=True)
season_cols = [c for c in df.columns if c.startswith('season_')]
FEATURE_COLS += season_cols

TARGET = 'sm_tgt'

print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM (RBF approx)': Pipeline([
        ('scaler', StandardScaler()),
        ('nystroem', Nystroem(kernel='rbf', n_components=100, random_state=42)),
        ('sgd', SGDRegressor(random_state=42, max_iter=1000))
    ]),
    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
    ])
}

## Evaluation Function

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    results = {
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'train_r2': r2_score(y_train, y_train_pred),
        'test_r2': r2_score(y_test, y_test_pred),
        'test_bias': np.mean(y_test_pred - y_test),
    }
    return results, y_test_pred

## Experiment 1: Random 80/20 Split

In [ ]:
X = df[FEATURE_COLS].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

all_results = []

for name, model in models.items():
    print(f"Training {name}...")
    results, _ = evaluate_model(clone(model), X_train, y_train, X_test, y_test)
    results['model'] = name
    results['split'] = 'Random 80/20'
    all_results.append(results)
    print(f"  Test RMSE: {results['test_rmse']:.4f}, Test R2: {results['test_r2']:.4f}")

print("\nRandom split complete.")

## Experiment 2: West/East Spatial Split

In [ ]:
west_mask = df['region'] == 'West'
east_mask = df['region'] == 'East'

X_west = df.loc[west_mask, FEATURE_COLS].values
y_west = df.loc[west_mask, TARGET].values
X_east = df.loc[east_mask, FEATURE_COLS].values
y_east = df.loc[east_mask, TARGET].values

print(f"West (train): {X_west.shape[0]}, East (test): {X_east.shape[0]}")

spatial_predictions = df.loc[east_mask, ['latitude', 'longitude']].copy()

for name, model in models.items():
    print(f"Training {name} (West->East)...")
    results, y_pred = evaluate_model(clone(model), X_west, y_west, X_east, y_east)
    results['model'] = name
    results['split'] = 'West->East'
    all_results.append(results)
    spatial_predictions[f'pred_{name}'] = y_pred
    print(f"  Test RMSE: {results['test_rmse']:.4f}, Test R2: {results['test_r2']:.4f}")

spatial_predictions['actual'] = y_east
print("\nWest->East split complete.")

## Experiment 3: Spatial Block Cross-Validation

In [ ]:
blocks = df['spatial_block'].unique()
print(f"Blocks: {blocks}")

for name, model_template in models.items():
    print(f"\nBlock CV for {name}...")
    block_results = []

    for block in blocks:
        test_mask = df['spatial_block'] == block
        train_mask = ~test_mask

        X_tr = df.loc[train_mask, FEATURE_COLS].values
        y_tr = df.loc[train_mask, TARGET].values
        X_te = df.loc[test_mask, FEATURE_COLS].values
        y_te = df.loc[test_mask, TARGET].values

        if len(X_te) == 0:
            continue

        model = clone(model_template)
        res, _ = evaluate_model(model, X_tr, y_tr, X_te, y_te)
        block_results.append(res)
        print(f"  Block {block}: RMSE={res['test_rmse']:.4f}, R2={res['test_r2']:.4f}")

    avg_results = {
        'train_rmse': np.mean([r['train_rmse'] for r in block_results]),
        'test_rmse': np.mean([r['test_rmse'] for r in block_results]),
        'train_r2': np.mean([r['train_r2'] for r in block_results]),
        'test_r2': np.mean([r['test_r2'] for r in block_results]),
        'test_bias': np.mean([r['test_bias'] for r in block_results]),
        'model': name,
        'split': 'Spatial Block CV'
    }
    all_results.append(avg_results)
    print(f"  Average: RMSE={avg_results['test_rmse']:.4f}, R2={avg_results['test_r2']:.4f}")

## Save Results

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df[['model', 'split', 'train_rmse', 'test_rmse', 'train_r2', 'test_r2', 'test_bias']]
results_df.to_csv(f'{RESULTS_DIR}/model_results.csv', index=False)
print(results_df.to_string(index=False))

In [ ]:
spatial_predictions.to_csv(f'{RESULTS_DIR}/predictions_spatial.csv', index=False)
print(f"Spatial predictions saved: {spatial_predictions.shape}")

## Feature Importance (Random Forest)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

importance_df.to_csv(f'{RESULTS_DIR}/feature_importance.csv', index=False)
print(importance_df.to_string(index=False))